# Structured Scientific Data with HDF5: Design, Access, and Compression

In [ ]:
import h5py
import numpy as np
import time

# Structured Scientific Data with HDF5: Design, Access, and Compression

HDF5 is one of the most widely used scientific data formats. Many neuroscience tools, analysis pipelines, and standardised formats rely on it internally. In this notebook, we explore how HDF5 works at a practical level using the h5py library, which provides a NumPy-friendly interface to HDF5 files.

Our goals are to:

- Understand how to store structured data and metadata

- Read data efficiently and selectively

- Explore how compression affects disk usage and performance

- Throughout, we focus on design decisions: how we organise data, how we represent it, and how those choices influence performance.

## Setup

### Utility Functions

In [ ]:
import os
import psutil

def _format_bytes(bytes: float, precision: int = 2) -> str:
    """
    Takes a time in seconds and returns a string (e.g. ) that is more human-readable.

    Looking to do this in a real project?  Some alternatives:
      - `humanfriendly`: https://pypi.org/project/humanfriendly/#getting-started
    """

    if bytes < 0:
        raise ValueError("bytes must be non-negative")

    units = [("KB", 1000), ("MB", 1_000_000), ("GB", 1_000_000_000), ("TB", 1_000_000_000_000)]

    for unit, scale in reversed(units):
        if bytes >= scale:
            value = bytes / scale
            return f"{value:.{precision}f} {unit}"
    else:
        return f"{bytes} B"
    

def _disk_read() -> float:
    return psutil.Process(os.getpid()).io_counters().read_bytes

def _disk_write() -> float:
    return psutil.Process(os.getpid()).io_counters().write_bytes


class utils:
    format_bytes = _format_bytes
    bytes_read = _disk_read
    bytes_written = _disk_write

---

## Writing Mixed-Type, Labelled Data and Metadata to HDF5 Files using h5py

HDF5 allows us to move beyond simple array storage. Instead of writing one array per file, we can:
  - Store multiple datasets in a single file 
  - Organise data hierarchically using groups 
  - Attach metadata directly to datasets 
  - Combine numeric arrays with strings and other data types

This makes HDF5 suitable for structured scientific data, where measurements, metadata, and experimental parameters belong together.

In this section, we focus on:
  - Creating datasets
  - Organising them into groups
  - Adding metadata using attributes
  - Designing clear, interpretable file structures

The emphasis is not on complexity, but on clarity. We aim to create files that are both structured and understandable.

### Reference 

| Code | Description |
| :-- | :-- |
| `with h5py.File('temp.h5', 'w') as f:`<br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;`# write content in f` | Open an HDF5 file for writing *with* a context manager |
| `f = h5py.File('filename.h5', 'w')` | Open an h5py file object for writing *without* a context manager |
| `f.close()` | Closes the h5py file after writing *without* a context manager and releases the linked file back to the operating system.  |
| `f.create_dataset('temp', data=x)` | Write an array called 'temp' with the data in the numpy array `x` into the HDF5 file |
| `f.create_dataset('data/temp', data=x)` | Write an array called 'temp' in the folder called "data" with the data in the numpy array `x` into the HDF5 file |
| `f.attrs['name'] = 'Session 1'` | Set an attribute as metadata onto the root group of the HDF5 file -- this works like a normal Python dictionary |
| `f['x'].attrs['id'] = 'ABC'` | Set an attribute as metadata onto the 'x' node of the HDF file |



### Exercises

**Example**: Write an HDF5 File called `temp.h5` with the following schema:
```
root/
└── temp: uint16, 1000 x 1  (temperature measurements over time)
```

In [ ]:
temp = np.random.randint(15, 22, size=(1000,1)).astype(np.uint16)


In [ ]:
with h5py.File('temp.h5', 'w') as f:
    f.create_dataset('temp', data=temp)
    f['temp'].attrs.update({
        'description': 'Temperature measurements over time'
    })

**Exercise**: Open up this `temp.h5` file with the  [myHDF5 data browser](https://myhdf5.hdfgroup.org/) and explore the data in the GUI.  Does this look how you expected? 

![browser view](img/h5_1.png)

**Exercise**: Write an HDF5 File called `ephys.h5` with the following schema and descriptions:

```
root/
├── time: float32, 1 x 1000 (trial time, in seconds)
├── voltage: int16, 4 x 1000 (voltage measurements for each recording channel)
└── chan_names: S, 4 (channel names for each recording channel)
```

In [ ]:
t = np.linspace(0, 3, 1000).astype(np.float32)
voltage = np.random.normal(1, 1, size=(4, 1000)).astype(np.float32)
chan_names = ['CH01', 'CH02', 'CH03', 'CH05']

In [ ]:
with h5py.File('ephys.h5', 'w') as f:
    f.create_dataset('time', data=t)
    f['time'].attrs.update({
        'description': 'Trial time, in seconds.'
    })
    f.create_dataset('voltage', data=voltage)
    f['voltage'].attrs.update({
        'description': 'voltage measurements for each recording channel.'
    })
    f.create_dataset('chan_names', data=chan_names)
    f['chan_names'].attrs.update({
        'description': 'channel names for each recording channel.'
    })


**Exercise**: Open up your `ephys.h5` file with the  [myHDF5 data browser](https://myhdf5.hdfgroup.org/) and explore your data in the GUI.  Does this look how you expected? 

![browser view](img/h5_2.png)

**Exercise**: Write an HDF5 File called `motion_tracking.h5` with the following schema 
```
root/
├── session_date: str
├── subject_id: str
├── camera: 
│   ├── black_noise_image: uint16, 640 x 640 x 3 (reference image taken with lights out)
│   ├── image_width: uint16
│   ├── image_height: uint16
│   ├── shutter_speed: uint16
│   └── aperture: float32
│
└── motion_tracking: 
    ├── time: uint32 1 x 3000 (session time, in milliseconds)
    ├── rb_pos: float32  2 x 3 x 3000 (XYZ coordinates of the center of each tracked rigid body)
    ├── rb_rot: float32  2 x 3 x 3000 (XYZ Euler rotations of each tracked rigid body)
    ├── xyz_names: str 1 x 3 (The spatial coordinate names)
    └── rb_names: str 1 x 2 (The name of each rigid body)


```

In [ ]:
session_date = '2024-04-22'
subject_id = 'AD11'
camera_black_noise_im = np.random.randint(0, 30, size=(640, 640, 3)).astype(np.uint16)
im_width = 640
im_height = 640
shutter_speed = 800
aperture = 2.8
motion_time = (np.arange(0, 1000, step=1/shutter_speed)[:3000] * 1000).astype(np.uint32)
rb_pos = np.random.random(size=(2, 3, 3000)).astype(np.float32)
rb_rot = np.random.random(size=(2, 3, 3000)).astype(np.float32)
xyz_names = ['X', 'Y', 'Z']
rb_names = ['head', 'tail_base']

In [ ]:
with h5py.File('motion_tracking.h5', 'w') as f:
    f.attrs.update({
        'session_date': session_date,
        'subject_id': subject_id,
    })
    f.create_dataset('camera/black_noise_image', data=camera_black_noise_im)
    f['camera'].attrs.update({
        'image_width': im_width,
        'image_height': im_height,
        'shutter_speed': shutter_speed,
        'aperture': aperture,
    })

    f.create_dataset('motion_tracking/time', data=motion_time)
    f.create_dataset('motion_tracking/rb_pos', data=rb_pos)
    f.create_dataset('motion_tracking/rb_rot', data=rb_rot)
    f.create_dataset('motion_tracking/rb_names', data=rb_names)
    f.create_dataset('motion_tracking/xyz_names', data=xyz_names)
    


**Exercise**: Open up your `motion_tracking.h5` file with the  [myHDF5 data browser](https://myhdf5.hdfgroup.org/) and explore your data in the GUI.  Does this look how you expected? 

![browser view](img/h5_3.png)

## Reading Data from HDF5 Files

One of the strengths of HDF5 is that we do not need to load an entire dataset into memory; without loading all the data, we can:
  - Inspect file structure
  - Read selected datasets
  - Access metadata
  - Load only slices of large arrays

In this section, we practise navigating HDF5 files and reading data efficiently. We treat HDF5 files as structured containers and learn how to explore them programmatically.

Our goal is to become comfortable:
  - Inspecting file contents
  - Accessing nested groups 
  - Reading partial data
  - Retrieving metadata stored as attributes


### Reference 

| **Code** | **Description** |
| :-- | :-- |
| `with h5py.File('temp.h5', 'r') as f:`<br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;`# read content in f` | Open an HDF5 file for reading *with* a context manager |
| `f = h5py.File('filename.h5', 'r')` | Open an h5py file object for reading *without* a context manager |
| `f.close()` | Closes the h5py file after reading *without* a context manager.  |
| `f.keys()` | See a list of datasets and groups at the root node |
| `f.attrs` | Get the dict-like attributes at the root node |
| `f.attrs['a']` | Get the 'a' attribute at the root node  |
| `f['x'][:]` | Read in the 'x' dataset as a numpy array |
| `f['x'][5:20]` | Read in a slice of the 'x' dataset as a numpy array |
| `f['x'].keys()` | See a list of datasets and groups at the 'x' node |
| `f['folder']['x']` | Get the 'x' dataset in the 'folder' group |
| `f['x'].attrs['key'] ` | Get attribute named 'key_name' in the 'x' dataset |
| `f['folder/x']` | (Alternative Syntax) Get the 'x' dataset in the 'folder' group |





### Exercises

Run cell below to create `ephys.h5` and `motion_tracking.h5` files if you didn't already create them in the previous section.

In [ ]:
t = np.linspace(0, 3, 1000).astype(np.float32)
voltage = np.random.normal(1, 1, size=(4, 1000)).astype(np.float32)
chan_names = ['CH01', 'CH02', 'CH03', 'CH05']

with h5py.File('ephys.h5', 'w') as f:
    f.create_dataset('time', data=t)
    f['time'].attrs.update({
        'description': 'Trial time, in seconds.'
    })
    f.create_dataset('voltage', data=voltage)
    f['voltage'].attrs.update({
        'description': 'voltage measurements for each recording channel.'
    })
    f.create_dataset('chan_names', data=chan_names)
    f['chan_names'].attrs.update({
        'description': 'channel names for each recording channel.'
    })

session_date = '2024-04-22'
subject_id = 'AD11'
camera_black_noise_im = np.random.randint(0, 30, size=(640, 640, 3)).astype(np.uint16)
im_width = 640
im_height = 640
shutter_speed = 800
aperture = 2.8
motion_time = (np.arange(0, 1000, step=1/shutter_speed)[:3000] * 1000).astype(np.uint32)
rb_pos = np.random.random(size=(2, 3, 3000)).astype(np.float32)
rb_rot = np.random.random(size=(2, 3, 3000)).astype(np.float32)
xyz_names = ['X', 'Y', 'Z']
rb_names = ['head', 'tail_base']

with h5py.File('motion_tracking.h5', 'w') as f:
    f.attrs.update({
        'session_date': session_date,
        'subject_id': subject_id,
    })
    f.create_dataset('camera/black_noise_image', data=camera_black_noise_im)
    f['camera'].attrs.update({
        'image_width': im_width,
        'image_height': im_height,
        'shutter_speed': shutter_speed,
        'aperture': aperture,
    })

    f.create_dataset('motion_tracking/time', data=motion_time)
    f.create_dataset('motion_tracking/rb_pos', data=rb_pos)
    f.create_dataset('motion_tracking/rb_rot', data=rb_rot)
    f.create_dataset('motion_tracking/rb_names', data=rb_names)
    f.create_dataset('motion_tracking/xyz_names', data=xyz_names)
    



**Example**: From the temperature file, read in only the last 5 temperature measurements as a numpy array.

In [ ]:
f = h5py.File('temp.h5')
temp = f['temp'][-5:, :]
f.close()
temp

array([[17],
       [18],
       [21],
       [21],
       [15]], dtype=uint16)

**Exercise**: From the `ephys.h5` file, read in the first 10 voltage measurements as a numpy array.

In [ ]:
f = h5py.File('ephys.h5')
volts = f['voltage'][:, :10]
f.close()
volts

array([[ 1.2808231 ,  1.0320328 ,  0.874629  , -1.3136902 ,  1.6197271 ,
         1.5784456 ,  1.4069831 ,  0.68616134, -1.199052  ,  1.3606339 ],
       [ 1.6216406 ,  0.01776242,  1.4770164 ,  1.2611846 ,  1.3048651 ,
         0.97558033,  1.4829894 ,  0.49732113,  1.5942357 ,  2.6485615 ],
       [ 2.3461616 ,  1.296176  ,  1.2673831 ,  1.1293796 ,  0.39923346,
         1.4304402 ,  1.2079097 ,  1.3690414 ,  0.55070674,  1.6116276 ],
       [ 1.7006097 , -1.6904101 ,  1.8017668 ,  1.993884  ,  1.6706394 ,
         0.34337127,  0.04884503,  2.2786667 ,  2.8244925 ,  1.3352139 ]],
      dtype=float32)

**Exercise**: From the `motion_tracking.h5` file, get the all the XYZ positions of the first rigid body during the recording.

In [ ]:
f = h5py.File('motion_tracking.h5')
pos = f['motion_tracking']['rb_pos'][0, :, :]
f.close()
pos

array([[0.95324266, 0.85540223, 0.45277697, ..., 0.6037633 , 0.90133774,
        0.39305928],
       [0.8117994 , 0.76180404, 0.6759377 , ..., 0.60035753, 0.48637435,
        0.93137646],
       [0.5795232 , 0.41464403, 0.06326026, ..., 0.9049264 , 0.8421143 ,
        0.55254143]], shape=(3, 3000), dtype=float32)

**Exercise**: from the `ephys.h5` file, load the name of the second recording channel.  (*Note*: Because HDF5 stores byte arrays for strings, to get it as a single string, it can help to use the  `.asstr()` method)

In [ ]:
f = h5py.File('ephys.h5')
f['chan_names'].asstr()[1]

'CH02'

**Example**: From the `ephys.h5` file, load the description of the voltage dataset

In [ ]:
f = h5py.File('ephys.h5')
description_voltage = f['voltage'].attrs['description']
f.close()
description_voltage

'voltage measurements for each recording channel.'

**Exercise**: From the motion tracking file, get the camera's shutter speed during the session

In [ ]:
f = h5py.File('motion_tracking.h5')
ss = f['camera'].attrs['shutter_speed']
f.close()
ss


np.int64(800)

---

## Compression in HDF5 — Trading CPU for Disk


When we store large datasets in HDF5, we can choose whether to compress them. Compression reduces file size and can potentially allow for faster disk reads (because less data is transferred), but it introduces trade-offs such as increased CPU usage during compression and decompression.

In this section, we will:

* Create datasets with and without compression
* Compare file sizes
* Measure write time and read time
* Observe CPU versus wall time trade-offs

### Reference

| Code                    | Description                      |
| :-- | :-- |
| `h5py.File()`           | Open or create an HDF5 file      |
| `create_dataset()`      | Create a dataset inside the file |
| `compression="gzip"`    | Enable gzip compression          |
| `compression_opts=`     | Set gzip compression level       |
| `dataset[:]`            | Read entire dataset              |
| `utils.bytes_written()` | Measure disk writes              |
| `array.nbytes`          | Measure array size in RAM        |



### Exercises


**Exercise**: "Writing an Uncompressed Dataset".  Please run the code below, and look at the printed performance.  Does disk usage closely match RAM size?


In [ ]:
data = np.arange(30_000_000, dtype=np.int64)
print("Size in RAM:", utils.format_bytes(data.nbytes))

dr0 = utils.bytes_written()
t0_wall = time.perf_counter()
t0_cpu = time.process_time()

with h5py.File("data.h5", "w") as f:
    f.create_dataset("data", data=data)

t1_wall = time.perf_counter()
t1_cpu = time.process_time()
dr1 = utils.bytes_written()

print("Disk written:", utils.format_bytes(dr1 - dr0))
print("Wall time:", t1_wall - t0_wall)
print("CPU time:", t1_cpu - t0_cpu)



Size in RAM: 240.00 MB
Disk written: 240.01 MB
Wall time: 0.2731432680011494
CPU time: 0.27302949899999973


**Exercise**: "Writing Structured Data with Gzip Compression". Please run the code below, and look at the printed performance.  Does disk usage closely match RAM size?  What changed from the uncompressed case?

In [ ]:
data = np.arange(30_000_000, dtype=np.int64)
print("Size in RAM:", utils.format_bytes(data.nbytes))


dr0 = utils.bytes_written()
t0_wall = time.perf_counter()
t0_cpu = time.process_time()

with h5py.File("data.h5", "w") as f:
    f.create_dataset(
        "data",
        data=data,
        compression="gzip"
    )

t1_wall = time.perf_counter()
t1_cpu = time.process_time()
dr1 = utils.bytes_written()

print("Disk written:", utils.format_bytes(dr1 - dr0))
print("Wall time:", t1_wall - t0_wall)
print("CPU time:", t1_cpu - t0_cpu)

Size in RAM: 240.00 MB
Disk written: 45.54 MB
Wall time: 5.272747287999664
CPU time: 5.244093512


**Exercise**: "Writing Random Noise with Gzip Compression". Please run the code below, and look at the printed performance.  Does disk usage closely match RAM size?  What changed from the structured case?  Is it always worth it to compress data?

In [ ]:
data = np.random.rand(30_000, 1_000)
print("Size in RAM:", utils.format_bytes(data.nbytes))



dr0 = utils.bytes_written()
t0_wall = time.perf_counter()
t0_cpu = time.process_time()

with h5py.File("data.h5", "w") as f:
    f.create_dataset(
        "data",
        data=data,
        compression="gzip"
    )

t1_wall = time.perf_counter()
t1_cpu = time.process_time()
dr1 = utils.bytes_written()

print("Disk written:", utils.format_bytes(dr1 - dr0))
print("Wall time:", t1_wall - t0_wall)
print("CPU time:", t1_cpu - t0_cpu)

Size in RAM: 240.00 MB
Disk written: 226.75 MB
Wall time: 20.424173425002664
CPU time: 20.411430742999997


---

## How Many Bits Do We Need?

Data type is not an implementation detail — it is a design choice.

The number of bits we allocate determines:

  - Range of representable values
  - Precision of numerical values
  - Memory usage
  - Disk size
  - Compression behaviour

In many scientific workflows, default data types (such as float64) are used automatically. However, this may be unnecessary and costly. In this section, we explore integer and floating-point limits and learn how to safely reduce storage size without compromising data integrity.

We will:

  - Inspect value ranges of integer types
  - Examine floating-point precision limits
  - Use astype() to reduce storage size
  - Verify correctness using NumPy testing utilities

### Exercises

#### Values, Precision, and Memory Size of Data Types

**Exercise**: How much space do these data types take up, in bytes?  Use `np.dtype().itemsize` to print the byte size of:
  - 16-bit floats (`np.float16`)
  - 32-bit floats (`np.float32`)
  - 64-bit floats (`np.float64`)
  
How many bytes do they take up?  Are they different?

Code snippet: `np.dtype(np.float16).itemsize`

In [ ]:
np.dtype(np.float16).itemsize, np.dtype(np.float32).itemsize, np.dtype(np.float64).itemsize

(2, 4, 8)

**Exercise**: How much space do these data types take up, in bytes?  Use `np.dtype().itemsize` to print the byte size of:
  - 16-bit floats (`np.float16`)
  - 16-bit ints (`np.int16`)
  - 16-bit unsigned ints (`np.uint16`)
  
How many bytes do they take up?  Are they different?


In [ ]:
np.dtype(np.float16).itemsize, np.dtype(np.int16).itemsize, np.dtype(np.uint16).itemsize

(2, 2, 2)

**Exercise**: Boolean values are sometimes surprising in how they are stored in memory by Numpy; because they only store `True` and `False` values, by rights they should only take up 1 bit (1/8 of a byte).  Let's check if that's true.  How much space do these data types take up, in bytes?  Use `np.dtype().itemsize` to print the byte size of:
  - bools (`np.bool_`)
  - 8-bit ints (`np.int8`)
  
How many bytes do they take up?  Are they different from each other?


In [ ]:
np.dtype(np.bool_).itemsize, np.dtype(np.int8).itemsize

(1, 1)

**Exercise**: What values can `unsigned integers` hold?  Is there a big difference between, say, an 8-bit integer and a 32-bit integer?  Print the output of `np.iinfo()` and Compare `np.uint8`, `np.uint16`, `np.uint32`, and `np.uint64` and examine the minimum and maximum values for those data types.

*Code Snippet*: `print(np.iinfo(np.uint8))`

In [ ]:
print(np.iinfo(np.uint8))
print(np.iinfo(np.uint16))
print(np.iinfo(np.uint32))
print(np.iinfo(np.uint64))

Machine parameters for uint8
---------------------------------------------------------------
min = 0
max = 255
---------------------------------------------------------------

Machine parameters for uint16
---------------------------------------------------------------
min = 0
max = 65535
---------------------------------------------------------------

Machine parameters for uint32
---------------------------------------------------------------
min = 0
max = 4294967295
---------------------------------------------------------------

Machine parameters for uint64
---------------------------------------------------------------
min = 0
max = 18446744073709551615
---------------------------------------------------------------



**Exercise**: What values can `signed integers` hold?  Print the output of `np.iinfo()` and Compare `np.int8`, `np.int16`, `np.int32`, and `np.int64`.

What makes the values different than their corresponeding unsigned values? 

In [ ]:
print(np.iinfo(np.int8))
print(np.iinfo(np.int16))
print(np.iinfo(np.int32))
print(np.iinfo(np.int64))

Machine parameters for int8
---------------------------------------------------------------
min = -128
max = 127
---------------------------------------------------------------

Machine parameters for int16
---------------------------------------------------------------
min = -32768
max = 32767
---------------------------------------------------------------

Machine parameters for int32
---------------------------------------------------------------
min = -2147483648
max = 2147483647
---------------------------------------------------------------

Machine parameters for int64
---------------------------------------------------------------
min = -9223372036854775808
max = 9223372036854775807
---------------------------------------------------------------



**Exercise**: What values can `floats` hold, and what is an attribute that makes them different from ints?  Print the output of `np.finfo()` and Compare `np.float16`, `np.float32`, and `np.float64`:

In [ ]:
print(np.finfo(np.float16))
print(np.finfo(np.float32))
print(np.finfo(np.float64))

Machine parameters for float16
---------------------------------------------------------------
precision = 3   resolution = 0.001
machep = -10   eps =        0.000977
negep =  -11   epsneg =     0.0004883
minexp = -14   tiny =       6.104e-05
maxexp = 16   max =        6.55e+04
nexp =   5   min =        -max
smallest_normal = 6.104e-05   smallest_subnormal = 6e-08
---------------------------------------------------------------

Machine parameters for float32
---------------------------------------------------------------
precision = 6   resolution = 1e-06
machep = -23   eps =        1.1920929e-07
negep =  -24   epsneg =     5.9604645e-08
minexp = -126   tiny =       1.1754944e-38
maxexp = 128   max =        3.4028235e+38
nexp =   8   min =        -max
smallest_normal = 1.1754944e-38   smallest_subnormal = 1e-45
---------------------------------------------------------------

Machine parameters for float64
---------------------------------------------------------------
precision = 15   

#### Reducing Data Size with `np.astype()`

**Example**: What is the smallest data type you'd like to use to store the array below, without losing important data?  Use `.astype()` to make a transformed version of the data, then use `npt.assert_equal()` to verify that the transformed data has the same values as the original data.

In [ ]:
temperature_c = np.random.randint(low=-15, high=35, size=10_000)
utils.format_bytes(temperature_c.nbytes)

'80.00 KB'

In [ ]:
temperature_c2 = temperature_c.astype(np.int8)
print(utils.format_bytes(temperature_c2.nbytes))
npt.assert_equal(temperature_c, temperature_c2)

10.00 KB


**Exercise**: What is the smallest data type you'd like to use to store the array below, without losing important data?  Use `.astype()` to make a transformed version of the data, then use `npt.assert_equal()` to verify that the transformed data has the same values as the original data.

In [ ]:
pixel_brightness = np.random.randint(low=0, high=256, size=10_000)
utils.format_bytes(pixel_brightness.nbytes)

'80.00 KB'

In [ ]:
pixel_brightness2 = pixel_brightness.astype(np.uint8)
print(utils.format_bytes(pixel_brightness2.nbytes))
npt.assert_equal(pixel_brightness, pixel_brightness2)

10.00 KB


**Exercise**: What is the smallest data type you'd like to use to store the array below, without losing important data?  Use `.astype()` to make a transformed version of the data, then use `npt.assert_equal()` to verify that the transformed data has the same values as the original data.

In [ ]:
binned_spike_counts = np.random.poisson(lam=4, size=10_000)
utils.format_bytes(binned_spike_counts.nbytes)

'80.00 KB'

In [ ]:
binned_spike_counts2 = binned_spike_counts.astype(np.uint8)
print(utils.format_bytes(binned_spike_counts.nbytes))
npt.assert_equal(binned_spike_counts, binned_spike_counts2)

80.00 KB


**Exercise**: What is the smallest data type you'd like to use to store the array below, without losing important data?  Use `.astype()` to make a transformed version of the data, then use `npt.assert_equal()` to verify that the transformed data has the same values as the original data.

In [ ]:
time_samples = np.arange(0, 10_000)
utils.format_bytes(time_samples.nbytes)

'80.00 KB'

In [ ]:
time_samples2 = time_samples.astype(np.uint16)
print(utils.format_bytes(time_samples2.nbytes))
npt.assert_equal(time_samples, time_samples2)

20.00 KB


**Exercise**: What is the smallest data type you'd like to use to store the array below, without losing important data?  Use `.astype()` to make a transformed version of the data, then use `npt.assert_equal()` to verify that the transformed data has the same values as the original data.

In [ ]:
head_velocities = np.random.randint(low=-100000, high=100000, size=10_000)
utils.format_bytes(head_velocities.nbytes)

'80.00 KB'

In [ ]:
head_velocities2 = head_velocities.astype(np.int32)
print(utils.format_bytes(head_velocities2.nbytes))
npt.assert_equal(head_velocities, head_velocities2)


40.00 KB


**Exercise**: What is the smallest data type you'd like to use to store the array below, without losing important data? Use `.astype()` to make a transformed version of the data, then verify the data's integrity with `npt.assert_allclose()`.  

*Note*: These are floats, so you'll have to rather use  `npt.assert_allclose()` to verify that the transformed data has the same values as the original data, and note that here, you should specify an "absolute tolerance" (`atol`) and a "relative tolerance" (`rtol`), to say how different the new data is allowed to look from the old data.

In [ ]:
time_seconds = np.arange(0, 200, .001)
utils.format_bytes(time_seconds.nbytes)

'1.60 MB'

In [ ]:
time_seconds2 = time_seconds.astype(np.float16)
print(utils.format_bytes(time_seconds2.nbytes))
npt.assert_allclose(time_seconds, time_seconds2, atol=0.001, rtol=0.1)

400.00 KB


**Exercise**: What is the smallest data type you'd like to use to store the array below, without losing important data?  Use `.astype()` to make a transformed version of the data, then verify the data's integrity with `npt.assert_allclose()`.

In [ ]:
firing_rates = np.random.exponential(scale=4, size=10_000)
utils.format_bytes(firing_rates.nbytes)

'80.00 KB'

In [ ]:
firing_rates2 = firing_rates.astype(np.float32)
print(utils.format_bytes(firing_rates2.nbytes))
npt.assert_allclose(firing_rates, firing_rates2, atol=firing_rates.min()/2, rtol=0.001)


40.00 KB
